# DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

_Classical ML_

**Clusters are dense crowds; lonely points are noise. No k needed.**

k-means needs you to pick the number of clusters and assumes round blobs.

     DBSCAN needs neither. It finds clusters as regions where points are packed densely together.

---

This notebook is a practice scaffold for the **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_iris

data = load_iris(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We'll run **DBSCAN** on two interleaving half-moons — a shape k-means can't separate. We do it in three steps: (1) make the data, (2) cluster by density, (3) read off the clusters, noise, and core points.

### Step 1 — Make the two-moons dataset

`make_moons` generates two interleaving crescent shapes with a little noise. They're not round blobs, so a centroid-based method like k-means would cut them wrong — but DBSCAN only cares whether points are densely packed, so the shape doesn't matter.

In [ ]:
from sklearn.datasets import make_moons

X, _ = make_moons(n_samples=300, noise=0.06, random_state=0)

### Step 2 — Cluster by density with DBSCAN

DBSCAN has two knobs: `eps` (how close two points must be to count as neighbours) and `min_samples` (how many neighbours a point needs to be a dense **core** point). It grows clusters outward from core points and labels anything in a sparse region as **noise** (label `-1`). `fit_predict` returns one cluster label per point.

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.2, min_samples=5)
labels = db.fit_predict(X)

### Step 3 — Read off clusters, noise, and core points

The label `-1` is reserved for noise, so we count distinct clusters by excluding it. We then tally how many points landed in each cluster and how many points DBSCAN flagged as **core** samples — the dense interior points that the clusters were grown from.

In [ ]:
# label -1 means noise; count distinct cluster ids excluding it
unique_labels = set(labels)
n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))

print("clusters found:", n_clusters)
print("noise points:", n_noise)
print("label values:", sorted(unique_labels))

for lab in sorted(unique_labels):
    count = int(np.sum(labels == lab))
    print("  cluster", lab, "->", count, "points")

print("core samples:", len(db.core_sample_indices_))

## Visualize the data & results

_Can density clustering find the Iris species and flag outliers, without being told how many clusters?_

### Step 1 — Standardise Iris and project to 2-D

The Iris flowers have four features, which we can't plot directly. We first **standardise** them (so no single measurement dominates by scale), then use PCA to compress them to the two most informative directions, giving us a 2-D layout we can scatter.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

iris = load_iris()
Xs = StandardScaler().fit_transform(iris.data)
X2 = PCA(n_components=2, random_state=0).fit_transform(Xs)

### Step 2 — Cluster the projection and colour the points

We run DBSCAN on the 2-D projection, then colour each point by its cluster. Noise points (label `-1`) are drawn red; the remaining clusters cycle through a colour palette. Watch whether density alone recovers the flower groups and flags a few outliers as noise — all without ever being told how many clusters to expect.

In [ ]:
from sklearn.cluster import DBSCAN

labels = DBSCAN(eps=0.6, min_samples=5).fit_predict(X2)

# label -1 is noise; draw it red, the rest by cluster color
palette = np.array(["#4ea1ff", "#7ee787", "#c89bff"])
colors = np.where(labels == -1, "#ff7b72", palette[labels % len(palette)])

plt.scatter(X2[:, 0], X2[:, 1], c=colors, s=20)
plt.title("DBSCAN on the Iris dataset (PCA to 2-D, eps 0.6, minPts 5)")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.show()